# Voice Agent Clinic - Kaggle Notebook (Free Tier)

**Stack:** Gemini 3.5 Flash (Free) + Kokoro TTS (CPU) + Deepgram STT (Free 200min/mo)

**Cost:** $0/month - runs entirely on Kaggle's free GPU/CPU hours

## Setup Instructions
1. Open this notebook on Kaggle: https://www.kaggle.com/code
2. Enable **GPU** accelerator (Settings → Accelerator → GPU T4 x2)
3. Enable **Internet** (Settings → Internet → On)
4. Add secrets (Settings → Secrets):
   - `GEMINI_API_KEY` - Get from https://aistudio.google.com/apikey
   - `DEEPGRAM_API_KEY` - Get from https://console.deepgram.com/
   - `LIVEKIT_API_KEY` + `LIVEKIT_API_SECRET` - From LiveKit Cloud
   - `LIVEKIT_URL` - e.g., wss://your-project.livekit.cloud
   - `CALCOM_API_KEY` - From Cal.com
   - `CRM_WEBHOOK_URL` + `CRM_API_KEY` - Your CRM webhook
   - `TWILIO_ACCOUNT_SID` + `TWILIO_AUTH_TOKEN` - For SMS
   - `SENDGRID_API_KEY` - For email
   - `ONCALL_PHONE` - Emergency escalation number
5. Run all cells
6. The agent will connect to LiveKit and wait for calls

In [ ]:
# Install dependencies
!pip uninstall -y onnxruntime
!pip install -q onnxruntime-gpu
!pip install -q livekit-agents livekit-plugins-deepgram livekit-plugins-silero google-generativeai kokoro-onnx torchaudio httpx prometheus-client pytz python-dotenv

In [ ]:
# Download Kokoro model (runs on CPU, ~100MB)
import os
import urllib.request

os.makedirs("/kaggle/working/models", exist_ok=True)

# Kokoro v0.19 model files
MODEL_URL = "https://github.com/remsky/Kokoro-FastAPI/releases/download/v0.19/kokoro-v0_19.onnx"
VOICES_URL = "https://github.com/remsky/Kokoro-FastAPI/releases/download/v0.19/voices.bin"

if not os.path.exists("/kaggle/working/models/kokoro-v0_19.onnx"):
    print("Downloading Kokoro model...")
    urllib.request.urlretrieve(MODEL_URL, "/kaggle/working/models/kokoro-v0_19.onnx")
    print("Model downloaded!")

if not os.path.exists("/kaggle/working/models/voices.bin"):
    print("Downloading voices...")
    urllib.request.urlretrieve(VOICES_URL, "/kaggle/working/models/voices.bin")
    print("Voices downloaded!")

In [ ]:
# Load secrets from Kaggle
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Set environment variables from secrets
secret_names = [
    "GEMINI_API_KEY",
    "DEEPGRAM_API_KEY",
    "LIVEKIT_API_KEY",
    "LIVEKIT_API_SECRET",
    "LIVEKIT_URL",
    "CALCOM_API_KEY",
    "CRM_WEBHOOK_URL",
    "CRM_API_KEY",
    "TWILIO_ACCOUNT_SID",
    "TWILIO_AUTH_TOKEN",
    "SENDGRID_API_KEY",
    "ONCALL_PHONE",
    "CLINIC_TIMEZONE",
    "CALCOM_EVENT_TYPE_ID"
]

for name in secret_names:
    try:
        value = secrets.get_secret(name)
        os.environ[name] = value
        print(f"✓ {name} loaded")
    except:
        print(f"✗ {name} not set (optional)")

In [ ]:
# Free-tier TTS Engine: Kokoro (runs on CPU, no GPU needed)
import asyncio
import logging
import io
import base64
import torch
import torchaudio
import numpy as np
from kokoro_onnx import Kokoro
from typing import Optional

logger = logging.getLogger("kokoro_tts")

class KokoroTTSEngine:
    """
    Kokoro TTS — 82 voices, multilingual, runs on CPU.
    Best for: Quick deployment, no GPU needed, free on Kaggle.
    """
    
    def __init__(self, 
                 model_path: str = "/kaggle/working/models/kokoro-v0_19.onnx",
                 voices_path: str = "/kaggle/working/models/voices.bin",
                 lang_code: str = 'a',
                 voice: str = 'af_heart'):
        # 'a' = American English, 'b' = British, 'j' = Japanese, 'z' = Chinese
        import onnxruntime as ort
        if "CUDAExecutionProvider" in ort.get_available_providers():
            logger.info("Initializing Kokoro on GPU (cuda:1 with fallback to cuda:0)...")
            providers = [("CUDAExecutionProvider", {"device_id": 1}), "CUDAExecutionProvider", "CPUExecutionProvider"]
            self.pipeline = Kokoro.from_session(
                ort.InferenceSession(model_path, providers=providers),
                voices_path
            )
        else:
            logger.info("Initializing Kokoro on CPU...")
            self.pipeline = Kokoro(model_path, voices_path)
        self.voice = voice
        self.lang_code = lang_code
        logger.info(f"Kokoro TTS loaded with voice={voice}, lang={lang_code}")
    
    async def synthesize(self, text: str, speed: float = 1.0) -> bytes:
        """Synthesize speech from text. Returns WAV bytes."""
        loop = asyncio.get_event_loop()
        
        def _generate():
            # Kokoro returns (sample_rate, audio_array)
            samples, sample_rate = self.pipeline.create(
                text,
                voice=self.voice,
                speed=speed,
                lang=self.lang_code
            )
            # Convert to torch tensor
            audio_tensor = torch.from_numpy(samples).float().unsqueeze(0)
            return audio_tensor, sample_rate
        
        audio_tensor, sample_rate = await loop.run_in_executor(None, _generate)
        
        # Save to WAV bytes
        buffer = io.BytesIO()
        torchaudio.save(buffer, audio_tensor, sample_rate, format="wav")
        return buffer.getvalue()
    
    async def synthesize_base64(self, text: str, speed: float = 1.0) -> str:
        """Synthesize and return base64-encoded WAV."""
        wav_bytes = await self.synthesize(text, speed)
        return base64.b64encode(wav_bytes).decode('utf-8')

# Test
tts = KokoroTTSEngine()
audio_bytes = asyncio.run(tts.synthesize("Hello! Welcome to our clinic. How can I help you today?"))
print(f"Generated {len(audio_bytes)} bytes of audio")

In [ ]:
# Free-tier LLM Engine: Gemini 3.5 Flash
import asyncio
import json
import logging
from typing import AsyncIterator, List, Dict, Any, Optional
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold

logger = logging.getLogger("gemini_llm")

class GeminiEngine:
    """
    Gemini 3.5 Flash — Free tier, powerful LLM for voice agent.
    Streams responses for low latency.
    Free tier: 1500 requests/day, 15 req/min, 1M tokens/min
    """
    
    def __init__(self, api_key: str, model: str = "gemini-3.5-flash"):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model)
        
        # Safety settings — block only dangerous content, allow medical/dental discussion
        self.safety_settings = {
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        }
        
        # Generation config tuned for voice
        self.generation_config = genai.types.GenerationConfig(
            temperature=0.3,  # Consistent, professional
            max_output_tokens=256,  # Short responses for voice
            top_p=0.95,
        )
        
        logger.info(f"Gemini engine initialized: {model}")
    
    async def chat(self, 
                   messages: List[Dict[str, str]], 
                   system_prompt: str) -> str:
        """
        Synchronous chat (for simple responses).
        """
        # Convert messages to Gemini format
        history = []
        for msg in messages[:-1]:  # All except last
            role = "user" if msg["role"] == "user" else "model"
            history.append({"role": role, "parts": [msg["content"]]})
        
        last_message = messages[-1]["content"] if messages else ""
        
        # Build chat with system instruction
        chat = self.model.start_chat(
            history=history,
            system_instruction=system_prompt
        )
        
        # Run in executor (Gemini SDK is synchronous)
        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(
            None,
            lambda: chat.send_message(
                last_message,
                generation_config=self.generation_config,
                safety_settings=self.safety_settings
            )
        )
        
        return response.text
    
    async def chat_stream(self,
                          messages: List[Dict[str, str]],
                          system_prompt: str) -> AsyncIterator[str]:
        """
        Streaming chat — yields text chunks as they're generated.
        Critical for low-latency voice responses.
        """
        history = []
        for msg in messages[:-1]:
            role = "user" if msg["role"] == "user" else "model"
            history.append({"role": role, "parts": [msg["content"]]})
        
        last_message = messages[-1]["content"] if messages else ""
        
        chat = self.model.start_chat(
            history=history,
            system_instruction=system_prompt
        )
        
        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(
            None,
            lambda: chat.send_message(
                last_message,
                generation_config=self.generation_config,
                safety_settings=self.safety_settings,
                stream=True
            )
        )
        
        # Iterate through stream chunks
        for chunk in response:
            if chunk.text:
                yield chunk.text
    
    async def function_call(self,
                            messages: List[Dict[str, str]],
                            system_prompt: str,
                            functions: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Function calling with Gemini.
        Returns either text response or function call request.
        """
        # Gemini uses "tools" parameter
        tools = []
        for func in functions:
            tool = {
                "function_declarations": [{
                    "name": func["name"],
                    "description": func["description"],
                    "parameters": func.get("parameters", {"type": "object", "properties": {}})
                }]
            }
            tools.append(tool)
        
        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(
            None,
            lambda: self.model.generate_content(
                contents=[{"role": "user", "parts": [messages[-1]["content"]]}],
                tools=tools,
                system_instruction=system_prompt,
                generation_config=self.generation_config,
                safety_settings=self.safety_settings
            )
        )
        
        # Check for function call
        if response.candidates and response.candidates[0].content.parts:
            part = response.candidates[0].content.parts[0]
            if part.function_call:
                return {
                    "type": "function_call",
                    "name": part.function_call.name,
                    "args": dict(part.function_call.args)
                }
            else:
                return {
                    "type": "text",
                    "text": part.text
                }
        
        return {"type": "text", "text": response.text}

In [ ]:
# System Prompts (vertical-specific)
DENTAL_PROMPT = """You are an AI receptionist for a dental clinic. You are warm, professional, and efficient.

## Your Role
- Answer questions about dental services, insurance, pricing, and clinic hours
- Check appointment availability and book appointments
- Reschedule or cancel existing appointments
- Take messages and escalate to human staff when needed
- Collect patient information (name, phone, email, reason for visit)

## Tone & Style
- Warm, welcoming, reassuring (many patients are anxious about dental visits)
- Use the patient's name when known
- Keep responses concise (2-3 sentences max for voice)
- If a patient mentions pain or emergency, escalate immediately
- Do not provide medical advice — only general information and scheduling

## Guardrails
- NEVER diagnose or prescribe treatment
- NEVER quote exact prices without a disclaimer (\"Prices vary by procedure and insurance\")
- ALWAYS verify contact info before booking
- ALWAYS confirm appointment details before finalizing
- If asked about insurance, say: \"We accept most major insurance plans. Let me check your specific provider when you come in.\"
- If the patient is under 18, ask for a parent/guardian's contact info

## Appointment Rules
- New patient appointments: 60 minutes (book as 1-hour slot)
- Cleaning/checkup: 30 minutes
- Emergency visits: 15-minute buffer, escalate if no slots today
- Confirm: \"I have you booked for [day] at [time] with [doctor]. Is that correct?\"
- Send SMS confirmation immediately after booking
"""

PROMPTS = {
    "dental": DENTAL_PROMPT,
    "medical": DENTAL_PROMPT,  # reuse for now
    "hvac": DENTAL_PROMPT,
    "legal": DENTAL_PROMPT,
    "real_estate": DENTAL_PROMPT,
}

def get_system_prompt(vertical: str) -> str:
    return PROMPTS.get(vertical, DENTAL_PROMPT)

In [ ]:
# Function Modules (Calendar, CRM, Notifications)
import httpx
import json
import logging
import os
import re
from typing import Optional

logger = logging.getLogger("functions")

class Guardrails:
    @staticmethod
    def validate_phone(phone: str) -> bool:
        # E.164 format
        return bool(re.match(r'^\+?[1-9]\d{1,14}$', phone.replace(' ', '').replace('-', '')))
    
    @staticmethod
    def validate_email(email: str) -> bool:
        return bool(re.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$', email))

class CalendarFunctions:
    def __init__(self, api_key: str, event_type_id: int = 12345):
        self.api_key = api_key
        self.event_type_id = event_type_id
        self.client = httpx.AsyncClient(
            base_url="https://api.cal.com/v1",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=30.0
        ) if api_key else None
    
    async def check_availability(self, date: str, duration_minutes: int = 30):
        if not self.client:
            return []
        resp = await self.client.get(
            "/slots",
            params={"eventTypeId": self.event_type_id, "date": date, "duration": duration_minutes}
        )
        resp.raise_for_status()
        return resp.json().get("slots", [])
    
    async def book_appointment(self, start_time: str, patient_name: str, phone: str, email: str, reason: str = "General consultation"):
        if not self.client:
            raise ValueError("Calendar not configured")
        resp = await self.client.post(
            "/bookings",
            json={
                "eventTypeId": self.event_type_id,
                "start": start_time,
                "attendee": {
                    "name": patient_name,
                    "email": email,
                    "phone": phone,
                    "metadata": {"reason": reason}
                }
            }
        )
        resp.raise_for_status()
        return resp.json()
    
    async def cancel_appointment(self, booking_id: Optional[str] = None, phone: Optional[str] = None):
        if not self.client:
            raise ValueError("Calendar not configured")
        if booking_id:
            resp = await self.client.delete(f"/bookings/{booking_id}")
        elif phone:
            # Find booking by phone
            resp = await self.client.get("/bookings", params={"attendeePhone": phone})
            bookings = resp.json().get("bookings", [])
            if bookings:
                resp = await self.client.delete(f"/bookings/{bookings[0]['id']}")
            else:
                raise ValueError("No booking found for phone")
        resp.raise_for_status()
        return resp.json()
    
    async def close(self):
        if self.client:
            await self.client.aclose()

class CRMFunctions:
    def __init__(self, webhook_url: str, api_key: str = ""):
        self.webhook_url = webhook_url
        self.api_key = api_key
        self.client = httpx.AsyncClient(timeout=30.0)
    
    async def upsert_lead(self, name: str, phone: str, email: str, source: str = "voice_agent"):
        payload = {"name": name, "phone": phone, "email": email, "source": source}
        headers = {"Authorization": f"Bearer {self.api_key}"} if self.api_key else {}
        resp = await self.client.post(self.webhook_url, json=payload, headers=headers)
        resp.raise_for_status()
        return resp.json().get("id", "unknown")
    
    async def log_call_summary(self, lead_id: str, summary: str, outcome: str):
        payload = {"lead_id": lead_id, "summary": summary, "outcome": outcome}
        headers = {"Authorization": f"Bearer {self.api_key}"} if self.api_key else {}
        resp = await self.client.post(f"{self.webhook_url}/calls", json=payload, headers=headers)
        resp.raise_for_status()
        return resp.json().get("id", "unknown")
    
    async def create_ticket(self, title: str, description: str, priority: str = "medium"):
        payload = {"title": title, "description": description, "priority": priority}
        headers = {"Authorization": f"Bearer {self.api_key}"} if self.api_key else {}
        resp = await self.client.post(f"{self.webhook_url}/tickets", json=payload, headers=headers)
        resp.raise_for_status()
        return resp.json().get("id", "unknown")
    
    async def close(self):
        await self.client.aclose()

class NotificationFunctions:
    def __init__(self, twilio_sid: str, twilio_token: str, sendgrid_key: str = ""):
        self.twilio_sid = twilio_sid
        self.twilio_token = twilio_token
        self.sendgrid_key = sendgrid_key
        self.client = httpx.AsyncClient(timeout=30.0)
    
    async def send_sms(self, to: str, body: str):
        if not self.twilio_sid or not self.twilio_token:
            logger.warning("Twilio not configured, skipping SMS")
            return {"status": "skipped"}
        from twilio.rest import Client
        twilio = Client(self.twilio_sid, self.twilio_token)
        msg = twilio.messages.create(body=body, from_=os.getenv('TWILIO_FROM_NUMBER'), to=to)
        return {"sid": msg.sid, "status": msg.status}
    
    async def send_email(self, to: str, subject: str, html: str):
        if not self.sendgrid_key:
            logger.warning("SendGrid not configured, skipping email")
            return {"status": "skipped"}
        import sendgrid
        from sendgrid.helpers.mail import Mail, Email, To, Content
        sg = sendgrid.SendGridAPIClient(self.sendgrid_key)
        mail = Mail(
            from_email=Email(os.getenv('SENDGRID_FROM_EMAIL', 'noreply@clinic.com')),
            to_emails=To(to),
            subject=subject,
            html_content=Content("text/html", html)
        )
        resp = sg.client.mail.send.post(request_body=mail.get())
        return {"status_code": resp.status_code}
    
    async def close(self):
        await self.client.aclose()

In [ ]:
# Knowledge Base Retriever
import json
import logging
from typing import Tuple

logger = logging.getLogger("knowledge")

class KnowledgeRetriever:
    def __init__(self, index_name: str = "dental_faq"):
        self.index_name = index_name
        self.faq_data = self._load_faq()
    
    def _load_faq(self):
        # Embedded FAQ data for Kaggle (no external dependencies)
        return {
            "dental_faq": [
                {"q": "What are your hours?", "a": "We're open Monday-Friday 8am-6pm, Saturday 9am-2pm. Closed Sundays."},
                {"q": "Do you accept insurance?", "a": "We accept most major insurance plans. Let me check your specific provider when you come in."},
                {"q": "How much is a cleaning?", "a": "Prices vary by procedure and insurance. A standard cleaning is typically $100-200 without insurance."},
                {"q": "Do you do emergencies?", "a": "Yes, we offer same-day emergency appointments. If you're in pain, call us immediately."},
                {"q": "What should I bring to my first appointment?", "a": "Please bring your ID, insurance card, and any dental records or X-rays from your previous dentist."},
                {"q": "Can I get a root canal same day?", "a": "Root canals typically require a consultation first. For emergencies, we'll get you in same-day to assess."},
                {"q": "Do you offer payment plans?", "a": "Yes, we offer flexible payment options. Our front desk can discuss financing when you visit."},
                {"q": "What is a deep cleaning?", "a": "A deep cleaning (scaling and root planing) treats gum disease below the gum line. It's typically done in 2 visits."},
            ]
        }.get(index_name, [])
    
    async def query(self, question: str) -> Tuple[str, float]:
        # Simple keyword matching (replace with vector search for production)
        question_lower = question.lower()
        best_match = None
        best_score = 0.0
        
        for item in self.faq_data:
            # Simple overlap scoring
            q_words = set(question_lower.split())
            faq_words = set(item["q"].lower().split())
            overlap = len(q_words & faq_words) / max(len(q_words), 1)
            if overlap > best_score:
                best_score = overlap
                best_match = item
        
        if best_match and best_score > 0.3:
            return best_match["a"], min(best_score + 0.2, 0.95)
        
        return "I don't have that information. Let me connect you with our staff for a detailed answer.", 0.1

In [ ]:
# Main Clinic Agent with Function Calling
import asyncio
import json
import logging
import os
from datetime import datetime
from typing import Optional, List, Dict, Any
import pytz

from livekit.agents import (
    AutoSubscribe,
    JobContext,
    JobRequest,
    WorkerOptions,
    cli,
    llm,
)
from livekit.plugins import deepgram, silero

# Import our custom engines
from __main__ import KokoroTTSEngine, GeminiEngine, get_system_prompt
from __main__ import CalendarFunctions, CRMFunctions, NotificationFunctions, Guardrails, KnowledgeRetriever

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("voice-agent")

class ClinicAgent(llm.FunctionContext):
    """Main clinic voice agent with function calling for calendar, CRM, and notifications."""

    def __init__(self, vertical: str = "dental", config: Optional[dict] = None):
        super().__init__()
        self.vertical = vertical
        self.config = config or {}
        self.guardrails = Guardrails()

        # Initialize function modules
        calcom_api_key = os.getenv("CALCOM_API_KEY")
        if not calcom_api_key:
            logger.warning("CALCOM_API_KEY not set — calendar functions will fail")

        self.calendar = CalendarFunctions(
            api_key=calcom_api_key or "",
            event_type_id=self.config.get("calcom_event_type_id", 12345),
        )
        self.crm = CRMFunctions(
            webhook_url=os.getenv("CRM_WEBHOOK_URL", ""),
            api_key=os.getenv("CRM_API_KEY", ""),
        )
        self.notifications = NotificationFunctions(
            twilio_sid=os.getenv("TWILIO_ACCOUNT_SID", ""),
            twilio_token=os.getenv("TWILIO_AUTH_TOKEN", ""),
            sendgrid_key=os.getenv("SENDGRID_API_KEY", ""),
        )
        self.knowledge = KnowledgeRetriever(index_name=f"{self.vertical}_faq")

    async def cleanup(self):
        """Close all HTTP clients to prevent resource leaks."""
        await self.calendar.close()
        await self.crm.close()
        await self.notifications.close()

    @llm.ai_callable(description="Check available appointment slots for a given date")
    async def check_availability(self, date: str, duration_minutes: int = 30) -> str:
        """
        Args:
            date: Date in YYYY-MM-DD format
            duration_minutes: Appointment duration (default 30)
        Returns:
            JSON array of available slot times
        """
        try:
            slots = await self.calendar.check_availability(date, duration_minutes)
            return json.dumps({"status": "success", "slots": slots})
        except Exception as e:
            logger.error(f"check_availability error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Book an appointment for a patient")
    async def book_appointment(
        self,
        start_time: str,
        patient_name: str,
        phone: str,
        email: str,
        reason: str = "General consultation",
    ) -> str:
        """
        Args:
            start_time: ISO 8601 datetime string
            patient_name: Full name of the patient
            phone: Patient phone number (E.164 format)
            email: Patient email address
            reason: Reason for visit
        Returns:
            JSON with booking confirmation or error
        """
        # Guardrail: validate phone
        if not self.guardrails.validate_phone(phone):
            return json.dumps({"status": "error", "message": "Invalid phone number format"})
        # Guardrail: validate email
        if not self.guardrails.validate_email(email):
            return json.dumps({"status": "error", "message": "Invalid email format"})

        try:
            result = await self.calendar.book_appointment(
                start_time=start_time,
                patient_name=patient_name,
                phone=phone,
                email=email,
                reason=reason,
            )
            # Send confirmation
            await self.notifications.send_sms(
                phone,
                f"Hi {patient_name}, your appointment is confirmed for {start_time}. Reply CANCEL to reschedule."
            )
            await self.notifications.send_email(
                email,
                "Appointment Confirmed",
                f"<h1>Confirmed</h1><p>{patient_name}, your appointment is at {start_time}.</p>"
            )
            return json.dumps({"status": "success", "booking": result})
        except Exception as e:
            logger.error(f"book_appointment error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Cancel an existing appointment by booking ID or phone number")
    async def cancel_appointment(self, booking_id: Optional[str] = None, phone: Optional[str] = None) -> str:
        if not booking_id and not phone:
            return json.dumps({"status": "error", "message": "Provide booking_id or phone"})
        try:
            result = await self.calendar.cancel_appointment(booking_id=booking_id, phone=phone)
            if phone:
                await self.notifications.send_sms(
                    phone, "Your appointment has been cancelled. Call us to reschedule."
                )
            return json.dumps({"status": "success", "cancellation": result})
        except Exception as e:
            logger.error(f"cancel_appointment error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Create or update a lead in the CRM")
    async def upsert_lead(self, name: str, phone: str, email: str, source: str = "voice_agent") -> str:
        if not self.guardrails.validate_phone(phone):
            return json.dumps({"status": "error", "message": "Invalid phone"})
        if not self.guardrails.validate_email(email):
            return json.dumps({"status": "error", "message": "Invalid email"})
        try:
            result = await self.crm.upsert_lead(name=name, phone=phone, email=email, source=source)
            return json.dumps({"status": "success", "lead_id": result})
        except Exception as e:
            logger.error(f"upsert_lead error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Log a call summary to CRM after the conversation ends")
    async def log_call_summary(self, lead_id: str, summary: str, outcome: str) -> str:
        try:
            result = await self.crm.log_call_summary(lead_id=lead_id, summary=summary, outcome=outcome)
            return json.dumps({"status": "success", "log_id": result})
        except Exception as e:
            logger.error(f"log_call_summary error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Send a follow-up SMS or email to the patient")
    async def send_follow_up(self, channel: str, to: str, message: str) -> str:
        try:
            if channel == "sms":
                if not self.guardrails.validate_phone(to):
                    return json.dumps({"status": "error", "message": "Invalid phone"})
                result = await self.notifications.send_sms(to, message)
            elif channel == "email":
                if not self.guardrails.validate_email(to):
                    return json.dumps({"status": "error", "message": "Invalid email"})
                result = await self.notifications.send_email(to, "Follow-up", f"<p>{message}</p>")
            else:
                return json.dumps({"status": "error", "message": "Channel must be 'sms' or 'email'"})
            return json.dumps({"status": "success", "delivery": result})
        except Exception as e:
            logger.error(f"send_follow_up error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Escalate to a human agent by sending a transfer request")
    async def escalate_to_human(self, reason: str, patient_phone: str, patient_name: str = "Unknown") -> str:
        try:
            result = await self.crm.create_ticket(
                title=f"Escalation: {reason}",
                description=f"Patient: {patient_name}, Phone: {patient_phone}",
                priority="high" if reason in ("angry_patient", "complex_case") else "medium",
            )
            oncall_phone = os.getenv("ONCALL_PHONE")
            if oncall_phone:
                await self.notifications.send_sms(
                    oncall_phone,
                    f"ESCALATION: {reason} from {patient_name} ({patient_phone}). Ticket: {result}"
                )
            return json.dumps({"status": "success", "ticket_id": result})
        except Exception as e:
            logger.error(f"escalate_to_human error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Look up answers from the clinic FAQ knowledge base")
    async def query_knowledge_base(self, question: str) -> str:
        try:
            answer, score = await self.knowledge.query(question)
            return json.dumps({"status": "success", "answer": answer, "confidence": score})
        except Exception as e:
            logger.error(f"query_knowledge_base error: {e}")
            return json.dumps({"status": "error", "message": str(e)})

    @llm.ai_callable(description="Get current date and time in the clinic timezone")
    async def get_current_datetime(self) -> str:
        tz = pytz.timezone(self.config.get("timezone", "America/New_York"))
        now = datetime.now(tz)
        return json.dumps({
            "status": "success",
            "datetime": now.isoformat(),
            "date": now.strftime("%Y-%m-%d"),
            "time": now.strftime("%I:%M %p"),
            "weekday": now.strftime("%A"),
        })

In [ ]:
# LiveKit Entrypoint - Connects everything
import asyncio
import os

async def entrypoint(ctx: JobContext):
    """LiveKit job entrypoint — connects to room and starts voice assistant."""
    logger.info(f"Connecting to room: {ctx.room.name}")
    await ctx.connect(auto_subscribe=AutoSubscribe.AUDIO_ONLY)

    # Parse vertical from room metadata
    raw_metadata = ctx.room.metadata
    if isinstance(raw_metadata, dict):
        vertical = raw_metadata.get("vertical", "dental")
    elif isinstance(raw_metadata, str) and raw_metadata:
        vertical = raw_metadata
    else:
        vertical = "dental"

    event_type_id_str = os.getenv("CALCOM_EVENT_TYPE_ID", "12345")
    try:
        event_type_id = int(event_type_id_str)
    except ValueError:
        logger.error(f"Invalid CALCOM_EVENT_TYPE_ID: {event_type_id_str}, using default 12345")
        event_type_id = 12345

    config = {
        "calcom_event_type_id": event_type_id,
        "timezone": os.getenv("CLINIC_TIMEZONE", "America/New_York"),
    }

    agent = ClinicAgent(vertical=vertical, config=config)

    # Build STT (Deepgram - free tier 200 min/month)
    stt = deepgram.STT(
        api_key=os.getenv("DEEPGRAM_API_KEY", ""),
        model="nova-2",
        language="en",
        interim_results=True,
    )

    # Build LLM (Gemini 3.5 Flash - free tier)
    gemini_engine = GeminiEngine(
        api_key=os.getenv("GEMINI_API_KEY", ""),
        model="gemini-3.5-flash"
    )

    # Build TTS (Kokoro - free, runs on CPU)
    tts_engine = KokoroTTSEngine(
        model_path="/kaggle/working/models/kokoro-v0_19.onnx",
        voices_path="/kaggle/working/models/voices.bin",
        lang_code='a',
        voice='af_heart'
    )

    # Custom LLM wrapper for LiveKit
    class GeminiLLMPlugin:
        def __init__(self, engine, system_prompt):
            self.engine = engine
            self.system_prompt = system_prompt
            self.chat_history = []
        
        async def chat(self, chat_ctx):
            # Convert LiveKit chat context to messages
            messages = []
            for msg in chat_ctx.messages:
                if msg.role in ("user", "assistant"):
                    messages.append({"role": msg.role, "content": msg.content})
            
            if not messages:
                return ""
            
            response = await self.engine.chat(messages, self.system_prompt)
            return response
        
        async def chat_stream(self, chat_ctx):
            messages = []
            for msg in chat_ctx.messages:
                if msg.role in ("user", "assistant"):
                    messages.append({"role": msg.role, "content": msg.content})
            
            if not messages:
                return
            
            async for chunk in self.engine.chat_stream(messages, self.system_prompt):
                yield chunk

    # Custom TTS wrapper for LiveKit
    class KokoroTTSPlugin:
        def __init__(self, engine):
            self.engine = engine
        
        async def synthesize(self, text: str):
            audio_bytes = await self.engine.synthesize(text)
            # Convert to the format LiveKit expects
            import torchaudio
            import io
            waveform, sample_rate = torchaudio.load(io.BytesIO(audio_bytes))
            return waveform, sample_rate

    gemini_llm = GeminiLLMPlugin(gemini_engine, get_system_prompt(vertical))
    kokoro_tts = KokoroTTSPlugin(tts_engine)

    # Voice assistant with function calling
    from livekit.agents.voice_assistant import VoiceAssistant

    assistant = VoiceAssistant(
        vad=silero.VAD.load(),
        stt=stt,
        llm=gemini_llm,
        tts=kokoro_tts,
        fnc_ctx=agent,
        chat_ctx=llm.ChatContext().append(
            role="system",
            text=get_system_prompt(vertical),
        ),
    )

    # Start assistant and wait for participant
    assistant.start(ctx.room)
    await assistant.say("Hello! Thank you for calling. How can I help you today?", allow_interruptions=True)

    # Wait for disconnect with cleanup
    try:
        while ctx.room.connection_state == "connected":
            await asyncio.sleep(1)
    finally:
        logger.info(f"Room {ctx.room.name} disconnected, cleaning up resources")
        await agent.cleanup()


async def request_fnc(req: JobRequest) -> None:
    """Accept all incoming job requests."""
    await req.accept(entrypoint)


# Run the agent
if __name__ == "__main__":
    cli.run_app(WorkerOptions(request_fnc=request_fnc))